# Post-Processing, Quality Metrics, and Manual Curation

Do Pre-processing on-the-fly (much faster)

In [ ]:
import spikeinterface.full as si

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

spikeglx_folder = Path('//path/Pixel1/1_auditory_neuropixels_BarakH/20240318_C9_T2_NP2_-10dB_g0')

stream_names, stream_ids = si.get_neo_streams('spikeglx', spikeglx_folder)

AP_rec = si.read_spikeglx(spikeglx_folder, stream_name='imec0.ap', load_sync_channel=False)

rec1 = si.highpass_filter(AP_rec, freq_min=400.)
bad_channel_ids, channel_labels = si.detect_bad_channels(rec1)
rec2 = rec1.remove_channels(bad_channel_ids)
print('bad_channel_ids', bad_channel_ids)

rec3 = si.phase_shift(rec2)
rec4 = si.common_reference(rec3, operator="median", reference="global")
rec = rec4
rec

Read Sorting

In [ ]:
## Load sorting
spikeglx_folder = Path('G:/20240312_C9_T1_NP2_-10dB_g0')
print(spikeglx_folder / 'kilosort2_5_output')

# the results can be read back for future session
sorting = si.read_sorter_folder(spikeglx_folder / 'kilosort2_5_output')
sorting

## Post Processing

In [ ]:
job_kwargs = dict(n_jobs=40, chunk_duration='1s', progress_bar=True)

analyzer = si.create_sorting_analyzer(sorting, rec, sparse=True, format="memory", **job_kwargs)
analyzer

In [ ]:
analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
analyzer.compute("waveforms", ms_before=1.5, ms_after=2., **job_kwargs)
analyzer.compute("templates", operators=["average", "median", "std"])
analyzer.compute("noise_levels")
analyzer.compute("correlograms")
analyzer.compute("unit_locations")
analyzer.compute("spike_amplitudes", **job_kwargs)
analyzer.compute("template_similarity")
analyzer

In [ ]:
analyzer_saved = analyzer.save_as(folder=spikeglx_folder / "analyzer", format="binary_folder")
analyzer_saved

## Quality metrics

In [ ]:
metric_names = ['firing_rate', 'presence_ratio', 'snr', 'isi_violation', 'amplitude_cutoff']

# metrics = analyzer.compute("quality_metrics").get_data()
# equivalent to
metrics = si.compute_quality_metrics(analyzer, metric_names=metric_names)

metrics

### Curation using metrics

In [ ]:
amplitude_cutoff_thresh = 0.1
isi_violations_ratio_thresh = 1
presence_ratio_thresh = 0.9

our_query = f"(amplitude_cutoff < {amplitude_cutoff_thresh}) & (isi_violations_ratio < {isi_violations_ratio_thresh}) & (presence_ratio > {presence_ratio_thresh})"
print(our_query)

In [ ]:
keep_units = metrics.query(our_query)
keep_unit_ids = keep_units.index.values
keep_unit_ids

## Export final results to disk folder and visulize with sortingview

In [ ]:
analyzer_clean = analyzer.select_units(keep_unit_ids, folder=spikeglx_folder / 'analyzer_clean', format='binary_folder')
analyzer_clean

In [ ]:
# export spike sorting report to a folder
si.export_report(analyzer_clean, spikeglx_folder / 'report', format='png', n_jobs=40, chunk_duration='1s',
                 progress_bar=True)
si.plot_sorting_summary(analyzer_clean, backend='sortingview')

### Manual Curation

In [ ]:
# si.plot_sorting_summary(analyzer_clean, backend='sortingview',curation=True)
# we open the printed link URL in a browser
# - make manual merges and labeling
# - from the curation box, click on "Save as snapshot (sha1://)"

In [ ]:
# from spikeinterface.curation import apply_sortingview_curation

# # copy the uri
# sha_uri = "sha1://59feb326204cf61356f1a2eb31f04d8e0177c4f1"
# clean_sorting = apply_sortingview_curation(sorting=sorting, uri_or_json=sha_uri)